In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from google.colab import drive

# 1. Kết nối Drive và tải dữ liệu v6
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/AI/south_vn_dht22_mq2_butane_fire_risk_v6_anomaly_clean.csv'
df = pd.read_csv(file_path)

# 2. Cấu hình đặc trưng và kích thước nhóm
features = ['temperature_C', 'mq2_adc']
TIME_STEPS = 30 # Nhóm 30 mẫu = 30 giây

# 3. Chuẩn hóa dữ liệu (Fit scaler chỉ trên tập Train)
train_df = df[df['split'] == 'train']
scaler = StandardScaler()
scaler.fit(train_df[features])

# Lưu Scaler lại cho máy Local dùng
joblib.dump(scaler, '/content/drive/MyDrive/AI/scaler_butane_v6.pkl')

# 4. Hàm đóng gói mỗi window_id (đúng 30 dòng) thành 1 mẫu phân tích
def process_split_fixed(split_name):
    split_df = df[df['split'] == split_name]
    X_seq, y_seq = [], []

    # Nhóm theo từng đợt thử nghiệm (window_id)
    for w_id, group in split_df.groupby('window_id'):
        if len(group) == TIME_STEPS:
            # Chuẩn hóa 30 dòng của nhóm này
            scaled_features = scaler.transform(group[features])
            label = group['label'].iloc[0] # Lấy nhãn đại diện

            X_seq.append(scaled_features)
            y_seq.append(label)

    return np.array(X_seq), np.array(y_seq)

print("Đang đóng gói dữ liệu thành các khối 30 giây...")
X_train, y_train = process_split_fixed('train')
X_val, y_val     = process_split_fixed('val')
X_test, y_test   = process_split_fixed('test')

print(f"Kích thước tập Train: {X_train.shape} (1260 nhóm, 30 dòng, 2 thông số)")
print(f"Kích thước tập Val:   {X_val.shape}")
print(f"Kích thước tập Test:  {X_test.shape}")

# 5. Xây dựng Mạng CNN 1D phân tích Động lực học (Tốc độ gia tăng)
model = Sequential([
    # kernel_size=5: Quét qua mỗi 5 giây để tính độ dốc nhiệt/gas
    Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=(TIME_STEPS, len(features))),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.4), # Tăng dropout lên 0.4 chống học vẹt
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 6. Huấn luyện AI
model.fit(X_train, y_train, epochs=25, batch_size=32, validation_data=(X_val, y_val))

# 7. Đánh giá và Lưu Model
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Độ chính xác trên tập Test: {acc*100:.2f}%")
model.save('/content/drive/MyDrive/AI/kitchen_fire_cnn1d_butane_v6.h5')

Mounted at /content/drive
Đang đóng gói dữ liệu thành các khối 30 giây...
Kích thước tập Train: (1260, 30, 2) (1260 nhóm, 30 dòng, 2 thông số)
Kích thước tập Val:   (270, 30, 2)
Kích thước tập Test:  (270, 30, 2)
Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9381 - loss: 0.2516 - val_accuracy: 1.0000 - val_loss: 0.0205
Epoch 2/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9992 - loss: 0.0097 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 3/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9992 - loss: 0.0042 - val_accuracy: 1.0000 - val_loss: 8.8871e-04
Epoch 4/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 1.0000 - val_loss: 5.6353e-04
Epoch 5/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 2.6543e-04
Epoch 6/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 3.0787e-04
Epoch 7/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 3.6166e-04 - val_accuracy: 1.0000 - val_loss: 4.4412e-04
Epoch 8/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 3.4543e-04 - val_accu


✅ Độ chính xác trên tập Test: 100.00%
